# cAPTure: benign-background audit by attack chain

This notebook determines which benign capture the authors declared when building each `train` attack chain. It also inspects `test` as a cross-check.

The audit is deliberately lightweight: it downloads the 14 official merge notebooks (not the multi-GB CSV or PCAP files), analyzes only their code cells, and searches for the `normal_...csv` file assignment. It produces a table, textual evidence, and SHA-256 hashes for the audited notebooks.

**What this establishes:** the benign provenance declared by the published pipeline. **What this does not establish:** that every published final CSV is the exact bit-for-bit output of that notebook version. That verification would require downloading the large CSV files.

In [ ]:
from pathlib import Path
import hashlib
import json
import re
import time

import pandas as pd
import requests
from IPython.display import display

BASE_DIR = Path('/content') if Path('/content').exists() else Path('/tmp')
WORKDIR = BASE_DIR / 'capture_benign_source_audit'
NOTEBOOK_DIR = WORKDIR / 'official_merge_notebooks'
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

# File IDs from cAPTure's public dataset_creation directory.
OFFICIAL_NOTEBOOKS = [
    {'chain': 'dollar_char', 'split': 'train', 'file_id': '1tw30woDFGbRaN4WFKOf_H5OJDPieLKqB'},
    {'chain': 'empty_conn',  'split': 'train', 'file_id': '1_DYpeEkJ-LSXijE_Nk4wLpe64xVUFF51'},
    {'chain': 'pub_exf',     'split': 'train', 'file_id': '1Cq3nxP4D1YIYazpCRxxwqan8RJX_MKrH'},
    {'chain': 'qos_mid',     'split': 'train', 'file_id': '1dskrOloCywp6JsuotN5QkpZKli2X8e8L'},
    {'chain': 'slash_char',  'split': 'train', 'file_id': '1oM7g9dJwO_Io_L-lDhQMse-MquKgDSx5'},
    {'chain': 'sub_exf',     'split': 'train', 'file_id': '1C-eY-icpV9bsvNDaRDDqxv5Rtndo9IaT'},
    {'chain': 'user_prop',   'split': 'train', 'file_id': '1fmbdrTHl0jmiqUldx3DAfnHjWE1AECgr'},
    {'chain': 'dollar_char', 'split': 'test',  'file_id': '1B4eeIno9ydjKFI1xlcTD0H4arKbZ2-tl'},
    {'chain': 'empty_conn',  'split': 'test',  'file_id': '1yrw1mJeWfpTvp9C2Qh2Lh0kuIbV3d0R8'},
    {'chain': 'pub_exf',     'split': 'test',  'file_id': '1kkGzLrxsJ7IN-aiz7Eifk5FDrVyumohg'},
    {'chain': 'qos_mid',     'split': 'test',  'file_id': '1607QX4zP7bwYYynwy5MuZK84g3Jb9W0X'},
    {'chain': 'slash_char',  'split': 'test',  'file_id': '12yTk-FMyr7-VE03L7CCq_K5EVPW1xfIv'},
    {'chain': 'sub_exf',     'split': 'test',  'file_id': '1dACthWk92BbXJUgLdY2rr1Srz-aDHDjB'},
    {'chain': 'user_prop',   'split': 'test',  'file_id': '1nyuThCoyWlAnwLQRO_Nk5twRVFnVZjnh'},
]

for item in OFFICIAL_NOTEBOOKS:
    item['notebook_name'] = f"3_merge_{item['chain']}_{item['split']}_to_normal.ipynb"

print(f'{len(OFFICIAL_NOTEBOOKS)} official notebooks will be audited.')

14 official notebooks will be audited.


In [ ]:
SOURCE_PATTERN = re.compile(r'normal_\d+(?:_\d+)+')

def _cell_source(cell):
    source = cell.get('source', '')
    return ''.join(source) if isinstance(source, list) else str(source)

def _is_notebook(payload):
    return isinstance(payload, dict) and isinstance(payload.get('cells'), list)

def download_official_notebook(item, force=False):
    target = NOTEBOOK_DIR / item['notebook_name']
    if target.exists() and not force:
        try:
            payload = json.loads(target.read_text(encoding='utf-8'))
            if _is_notebook(payload):
                return target, payload
        except Exception:
            pass

    file_id = item['file_id']
    urls = [
        f'https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t',
        f'https://drive.google.com/uc?export=download&id={file_id}',
    ]
    errors = []
    for attempt in range(3):
        for url in urls:
            try:
                response = requests.get(url, timeout=90)
                response.raise_for_status()
                payload = json.loads(response.content.decode('utf-8-sig'))
                if not _is_notebook(payload):
                    raise ValueError('the response does not have a valid notebook structure')
                target.write_bytes(response.content)
                return target, payload
            except Exception as exc:
                errors.append(f'{type(exc).__name__}: {exc}')
        time.sleep(2 ** attempt)

    raise RuntimeError(
        f"Could not download {item['notebook_name']} ({file_id}). "
        + ' | '.join(errors[-4:])
    )

def audit_notebook(item):
    path, payload = download_official_notebook(item)
    sources = set()
    evidence = []

    for cell_index, cell in enumerate(payload['cells']):
        if cell.get('cell_type') != 'code':
            continue
        for line_index, line in enumerate(_cell_source(cell).splitlines(), start=1):
            matches = SOURCE_PATTERN.findall(line)
            if matches:
                sources.update(matches)
                evidence.append({
                    'cell': cell_index,
                    'line': line_index,
                    'text': line.strip(),
                    'matches': sorted(set(matches)),
                })

    if len(sources) == 1:
        status = 'OK'
    elif not sources:
        status = 'MISSING'
    else:
        status = 'AMBIGUOUS'

    raw = path.read_bytes()
    return {
        'chain': item['chain'],
        'split': item['split'],
        'benign_capture': ' | '.join(sorted(sources)),
        'source_count': len(sources),
        'status': status,
        'notebook_name': item['notebook_name'],
        'google_drive_file_id': item['file_id'],
        'sha256': hashlib.sha256(raw).hexdigest(),
        'size_bytes': len(raw),
        'evidence': evidence,
    }

In [ ]:
rows = []
for index, item in enumerate(OFFICIAL_NOTEBOOKS, start=1):
    print(f"[{index:02d}/{len(OFFICIAL_NOTEBOOKS)}] {item['notebook_name']}")
    try:
        rows.append(audit_notebook(item))
    except Exception as exc:
        rows.append({
            'chain': item['chain'],
            'split': item['split'],
            'benign_capture': '',
            'source_count': 0,
            'status': f'DOWNLOAD_ERROR: {exc}',
            'notebook_name': item['notebook_name'],
            'google_drive_file_id': item['file_id'],
            'sha256': '',
            'size_bytes': 0,
            'evidence': [],
        })

report = pd.DataFrame(rows).sort_values(['split', 'chain'], ascending=[False, True]).reset_index(drop=True)
display(report[['chain', 'split', 'benign_capture', 'status', 'notebook_name', 'sha256']])

[01/14] 3_merge_dollar_char_train_to_normal.ipynb
[02/14] 3_merge_empty_conn_train_to_normal.ipynb
[03/14] 3_merge_pub_exf_train_to_normal.ipynb
[04/14] 3_merge_qos_mid_train_to_normal.ipynb
[05/14] 3_merge_slash_char_train_to_normal.ipynb
[06/14] 3_merge_sub_exf_train_to_normal.ipynb
[07/14] 3_merge_user_prop_train_to_normal.ipynb
[08/14] 3_merge_dollar_char_test_to_normal.ipynb
[09/14] 3_merge_empty_conn_test_to_normal.ipynb
[10/14] 3_merge_pub_exf_test_to_normal.ipynb
[11/14] 3_merge_qos_mid_test_to_normal.ipynb
[12/14] 3_merge_slash_char_test_to_normal.ipynb
[13/14] 3_merge_sub_exf_test_to_normal.ipynb
[14/14] 3_merge_user_prop_test_to_normal.ipynb


,chain,split,benign_capture,status,notebook_name,sha256
0,dollar_char,train,normal_15_16_17,OK,3_merge_dollar_char_train_to_normal.ipynb,a1c58c8867a65cebc472ec7258624ef9b38033ef67c3ec...
1,empty_conn,train,normal_2_3_4,OK,3_merge_empty_conn_train_to_normal.ipynb,bc06ee48de112a74aab1f1cd257048cdf778f7fa04ee0a...
2,pub_exf,train,normal_2_3_4,OK,3_merge_pub_exf_train_to_normal.ipynb,b6c18ae734f1b49b0258ae36f4ea805705f13ce868a434...
3,qos_mid,train,normal_2_3_4,OK,3_merge_qos_mid_train_to_normal.ipynb,9814dda27eb4f23473200f54f7dad73ee7c5ff7301121a...
4,slash_char,train,normal_15_16_17,OK,3_merge_slash_char_train_to_normal.ipynb,4748b4f77e9983f5e78bf23aea7346caa9eae31fb88281...
5,sub_exf,train,normal_15_16_17,OK,3_merge_sub_exf_train_to_normal.ipynb,b1aaa794b2c90516a73016459a859d1d4fbdd97ed74a88...
6,user_prop,train,normal_15_16_17,OK,3_merge_user_prop_train_to_normal.ipynb,49061a78291bcd136840f23eaa62008055b0d3834c3fcf...
7,dollar_char,test,normal_5_6_7,OK,3_merge_dollar_char_test_to_normal.ipynb,288d25da79b51f24b95c7199c757c213b3642dd98522e7...
8,empty_conn,test,normal_5_6_7,OK,3_merge_empty_conn_test_to_normal.ipynb,cccb345fc0e87117e6696fe94a0e97cc95ec79f6f49703...
9,pub_exf,test,normal_5_6_7,OK,3_merge_pub_exf_test_to_normal.ipynb,5f847788fe2d1d9be19ccbf189a7d7c6b58e7e9211dd99...


In [ ]:
expected_chains = {'dollar_char', 'empty_conn', 'pub_exf', 'qos_mid', 'slash_char', 'sub_exf', 'user_prop'}
problems = report[report['status'] != 'OK']
train_rows = report[report['split'] == 'train'].copy()
test_rows = report[report['split'] == 'test'].copy()
train_sources = sorted(set(train_rows.loc[train_rows['status'] == 'OK', 'benign_capture']))
test_sources = sorted(set(test_rows.loc[test_rows['status'] == 'OK', 'benign_capture']))
shared_sources = sorted(set(train_sources) & set(test_sources))

print('INTEGRITY CHECK')
print('--------------------')
print(f'Expected/observed rows: 14/{len(report)}')
print(f'Complete train chain set: {set(train_rows.chain) == expected_chains}')
print(f'Complete test chain set:  {set(test_rows.chain) == expected_chains}')
print(f'Notebooks with a non-unique result or an error: {len(problems)}')
print(f'Benign backgrounds found in train: {train_sources}')
print(f'Benign backgrounds found in test:  {test_sources}')
print(f'Benign backgrounds shared by train and test: {shared_sources}')

if len(problems):
    print('\nWARNING: do not interpret the mapping until these rows have been reviewed:')
    display(problems[['chain', 'split', 'status', 'notebook_name']])
else:
    print('\nTechnical result: each of the 14 notebooks contains exactly one recognizable benign reference.')

INTEGRITY CHECK
--------------------
Expected/observed rows: 14/14
Complete train chain set: True
Complete test chain set:  True
Notebooks with a non-unique result or an error: 0
Benign backgrounds found in train: ['normal_15_16_17', 'normal_2_3_4']
Benign backgrounds found in test:  ['normal_5_6_7']
Benign backgrounds shared by train and test: []

Technical result: each of the 14 notebooks contains exactly one recognizable benign reference.


In [ ]:
train_view = (
    train_rows[['chain', 'benign_capture', 'status', 'notebook_name', 'sha256']]
    .sort_values('chain')
    .rename(columns={
        'chain': 'attack_chain',
        'benign_capture': 'declared_benign_capture',
    })
)

print('TRAIN MAPPING (main result)')
display(train_view)


print(train_view[['attack_chain', 'declared_benign_capture', 'status']].to_csv(index=False))
print('Summary:')
print(f'train_sources={train_sources}')
print(f'test_sources={test_sources}')
print(f'shared_sources={shared_sources}')
print(f'problems={len(problems)}')


TRAIN MAPPING (main result)


,attack_chain,declared_benign_capture,status,notebook_name,sha256
0,dollar_char,normal_15_16_17,OK,3_merge_dollar_char_train_to_normal.ipynb,a1c58c8867a65cebc472ec7258624ef9b38033ef67c3ec...
1,empty_conn,normal_2_3_4,OK,3_merge_empty_conn_train_to_normal.ipynb,bc06ee48de112a74aab1f1cd257048cdf778f7fa04ee0a...
2,pub_exf,normal_2_3_4,OK,3_merge_pub_exf_train_to_normal.ipynb,b6c18ae734f1b49b0258ae36f4ea805705f13ce868a434...
3,qos_mid,normal_2_3_4,OK,3_merge_qos_mid_train_to_normal.ipynb,9814dda27eb4f23473200f54f7dad73ee7c5ff7301121a...
4,slash_char,normal_15_16_17,OK,3_merge_slash_char_train_to_normal.ipynb,4748b4f77e9983f5e78bf23aea7346caa9eae31fb88281...
5,sub_exf,normal_15_16_17,OK,3_merge_sub_exf_train_to_normal.ipynb,b1aaa794b2c90516a73016459a859d1d4fbdd97ed74a88...
6,user_prop,normal_15_16_17,OK,3_merge_user_prop_train_to_normal.ipynb,49061a78291bcd136840f23eaa62008055b0d3834c3fcf...


attack_chain,declared_benign_capture,status
dollar_char,normal_15_16_17,OK
empty_conn,normal_2_3_4,OK
pub_exf,normal_2_3_4,OK
qos_mid,normal_2_3_4,OK
slash_char,normal_15_16_17,OK
sub_exf,normal_15_16_17,OK
user_prop,normal_15_16_17,OK

Summary:
train_sources=['normal_15_16_17', 'normal_2_3_4']
test_sources=['normal_5_6_7']
shared_sources=[]
problems=0


In [ ]:
print('CODE-LINE EVIDENCE (TRAIN)')
print('--------------------------------------')
for row in sorted((r for r in rows if r['split'] == 'train'), key=lambda r: r['chain']):
    print(f"\n[{row['chain']}] {row['notebook_name']}")
    print(f"SHA-256: {row['sha256']}")
    if not row['evidence']:
        print('  (no evidence extracted)')
    for item in row['evidence']:
        print(f"  cell {item['cell']}, line {item['line']}: {item['text']}")

CODE-LINE EVIDENCE (TRAIN)
--------------------------------------

[dollar_char] 3_merge_dollar_char_train_to_normal.ipynb
SHA-256: a1c58c8867a65cebc472ec7258624ef9b38033ef67c3ecf17835b0409ab3080a
  cell 2, line 4: nrm_root = root + "normal/normal_simulated/normal_15_16_17/"
  cell 2, line 5: normal_csv = nrm_root + "normal_15_16_17.csv"

[empty_conn] 3_merge_empty_conn_train_to_normal.ipynb
SHA-256: bc06ee48de112a74aab1f1cd257048cdf778f7fa04ee0ad2fbf3e3fc346bffa5
  cell 2, line 4: nrm_root = root + "normal/normal_simulated/normal_2_3_4/"
  cell 2, line 5: normal_csv = nrm_root + "normal_2_3_4.csv"

[pub_exf] 3_merge_pub_exf_train_to_normal.ipynb
SHA-256: b6c18ae734f1b49b0258ae36f4ea805705f13ce868a4342e8a1b491cfff40435
  cell 2, line 4: nrm_root = root + "normal/normal_simulated/normal_2_3_4/"
  cell 2, line 5: normal_csv = nrm_root + "normal_2_3_4.csv"

[qos_mid] 3_merge_qos_mid_train_to_normal.ipynb
SHA-256: 9814dda27eb4f23473200f54f7dad73ee7c5ff7301121a5fe446c2f96d432f0e
  cell 2, l

In [ ]:
CSV_REPORT = WORKDIR / 'capture_benign_source_audit.csv'
JSON_EVIDENCE = WORKDIR / 'capture_benign_source_evidence.json'

report.drop(columns='evidence').to_csv(CSV_REPORT, index=False)
JSON_EVIDENCE.write_text(json.dumps(rows, indent=2, ensure_ascii=False), encoding='utf-8')

print(f'CSV report:      {CSV_REPORT}')
print(f'JSON evidence:   {JSON_EVIDENCE}')
print("You can download these files from Colab's Files panel.")

# To trigger browser downloads, change this to True and run this cell again.
DOWNLOAD_REPORTS_NOW = False
if DOWNLOAD_REPORTS_NOW:
    from google.colab import files
    files.download(str(CSV_REPORT))
    files.download(str(JSON_EVIDENCE))

CSV report:      /content/capture_benign_source_audit/capture_benign_source_audit.csv
JSON evidence:   /content/capture_benign_source_audit/capture_benign_source_evidence.json
You can download these files from Colab's Files panel.


## How to interpret the result

- If all 14 rows have `status == OK`, the extraction produced a unique result for every notebook.
- `declared_benign_capture` is the exact name found in the official code; it is not converted into a calendar date by inference.
- Different `train` chains using different benign backgrounds matters when designing `train/val/test`: the background may become correlated with chain identity.
- The hashes record the exact notebook versions that were audited.
- The published merge code selects a random insertion point. Therefore, regenerating the CSV files is unnecessary for this audit: identifying their declared provenance is sufficient. If data-level proof of the final CSV provenance is later required, a second signature-based audit can download and inspect one large file at a time.